# Calcolo Embedding E5-base — Kaggle (GPU T4)

**Prima di eseguire**: Settings → Accelerator → GPU T4 x2 → Save

### Flusso
1. Installa `sentence-transformers`
2. Carica `dataset_clean.csv` dal dataset Kaggle
3. Split temporale su TUTTI i ticket (nessun filtro area)
4. Calcola embedding `intfloat/multilingual-e5-base` su GPU
5. Salva in `/kaggle/working/` e scarica dalla UI (tab Output)

### Perché tutti i ticket e non solo quelli con area?
Gli embedding sono condivisi tra AreaClassifier e PriorityClassifier.
- PriorityClassifier usa tutti i ticket (inclusi quelli senza area)
- AreaClassifier filtra i ticket senza area **dopo** aver caricato gli embedding

Un unico set di embedding evita mismatch di dimensioni tra i due classificatori.

### File prodotti
```
e5_train.npy  →  embeddings/e5_train.npy
e5_test.npy   →  embeddings/e5_test.npy
```

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('ATTENZIONE: nessuna GPU — vai su Settings → Accelerator → GPU T4')

In [ ]:
# ── Caricamento CSV ───────────────────────────────────────────────────────────
# Carica il dataset_clean.csv caricato come dataset Kaggle.
# Per trovare il path corretto: nel pannello a sinistra apri il tab Data
# e copia il path del file (di solito /kaggle/input/<nome-dataset>/dataset_clean.csv)
CSV_PATH = '/kaggle/input/ticketclassifier-dataset/dataset_clean.csv'

SOGLIA_SPLIT = '2025-11-01'

df = pd.read_csv(CSV_PATH, parse_dates=['data_creazione'], engine='python')

# Split temporale — NESSUN filtro area: tutti i ticket
df_train = df[df['data_creazione'] <  SOGLIA_SPLIT].copy()
df_test  = df[df['data_creazione'] >= SOGLIA_SPLIT].copy()

print(f'Dataset totale : {len(df):,}')
print(f'Train          : {len(df_train):,}  (< {SOGLIA_SPLIT})')
print(f'Test           : {len(df_test):,}  (>= {SOGLIA_SPLIT})')
print()
print('NaN testo_input train:', df_train['testo_input'].isna().sum())
print('NaN testo_input test: ', df_test['testo_input'].isna().sum())

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'intfloat/multilingual-e5-base'

print(f'Carico modello {MODEL_NAME} su {device}...')
model = SentenceTransformer(MODEL_NAME, device=device)
print('Modello caricato.')
print()

# Il prefisso 'query: ' è richiesto dal modello E5 per testi di input.
# batch_size=256 funziona bene su T4 (16 GB); abbassa a 128 se vedi OOM.

print('[1/2] Calcolo embedding TRAIN...')
t0 = time.time()
emb_train = model.encode(
    ('query: ' + df_train['testo_input'].fillna('').astype(str)).tolist(),
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(f'TRAIN completato in {time.time()-t0:.0f}s — shape: {emb_train.shape}')
print()

print('[2/2] Calcolo embedding TEST...')
t0 = time.time()
emb_test = model.encode(
    ('query: ' + df_test['testo_input'].fillna('').astype(str)).tolist(),
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(f'TEST completato in {time.time()-t0:.0f}s — shape: {emb_test.shape}')

In [ ]:
# ── Salvataggio ───────────────────────────────────────────────────────────────
TRAIN_PATH = '/kaggle/working/e5_train.npy'
TEST_PATH  = '/kaggle/working/e5_test.npy'

np.save(TRAIN_PATH, emb_train)
np.save(TEST_PATH,  emb_test)

# Verifica integrità
assert np.allclose(np.load(TRAIN_PATH), emb_train), 'Errore: file train corrotto'
assert np.allclose(np.load(TEST_PATH),  emb_test),  'Errore: file test corrotto'

train_mb = os.path.getsize(TRAIN_PATH) / 1e6
test_mb  = os.path.getsize(TEST_PATH)  / 1e6

print(f'e5_train.npy — shape {emb_train.shape} — {train_mb:.0f} MB')
print(f'e5_test.npy  — shape {emb_test.shape}  — {test_mb:.0f} MB')
print()
print('Scarica i file dal tab Output nel pannello a destra.')
print('Poi copiali in: TicketClassifier/embeddings/')
print('  embeddings/e5_train.npy')
print('  embeddings/e5_test.npy')
print()
print('Questi file sono condivisi da AreaClassifier e PriorityClassifier.')
print('Ogni classificatore fa il proprio slice/filtro dopo il caricamento.')